In [4]:
# Cell 1: imports, find the raw file, load and inspect
import html
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

# ---- find the project root: the first folder (from here upwards) that contains "data" ----
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "data").exists():
    raise FileNotFoundError("No 'data' folder found at or above: " + str(Path.cwd()))

RAW_DIR = ROOT / "data" / "raw_data"
CLEAN_DIR = ROOT / "data" / "cleaned_data"
CLEAN_DIR.mkdir(exist_ok=True)
print("Project root:", ROOT)

for folder in (RAW_DIR, CLEAN_DIR):
    print(f"\nFiles in {folder.name}:")
    for name in sorted(os.listdir(folder)):
        print(f"  {name:<30}{(folder / name).stat().st_size / 1024:>10.1f} KB")

# ---- pick the raw file: the main file first, then the backup copies ----
candidates = ["jobs_raw.csv", "jobs_raw_backup2.csv", "jobs_raw_backup.csv"]
RAW_PATH = next((RAW_DIR / n for n in candidates if (RAW_DIR / n).exists()), None)
if RAW_PATH is None:
    raise FileNotFoundError("No raw file found in " + str(RAW_DIR) + ". Paste the file list printed above.")
print("\nUsing:", RAW_PATH.name)
if RAW_PATH.name != "jobs_raw.csv":
    print("WARNING: jobs_raw.csv is missing, so a backup copy is being used.")

raw = pd.read_csv(RAW_PATH, dtype={"job_id": str})

print("\nShape:", raw.shape)
print("\nColumn types:")
print(raw.dtypes)
print("\nMissing values per column:")
print(raw.isna().sum())
print("\nDuplicate job_ids:", raw["job_id"].duplicated().sum())
raw.head(3)

Project root: C:\Linkedin_Job_Analysis

Files in raw_data:
  jobs_cards.csv                      70.9 KB
  jobs_raw.csv                      1260.5 KB
  jobs_raw_backup.csv               1244.1 KB
  jobs_raw_backup2.csv              1258.4 KB

Files in cleaned_data:
  job_skills.csv                      63.6 KB
  jobs_cleaned.csv                  1306.4 KB

Using: jobs_raw.csv

Shape: (332, 15)

Column types:
job_id                 str
title                  str
company                str
location               str
posted_date            str
posted_text            str
job_url                str
seniority_level        str
employment_type        str
job_function       float64
industries             str
description            str
search_keyword         str
search_location        str
scraped_at             str
dtype: object

Missing values per column:
job_id               0
title                0
company              0
location             0
posted_date          0
posted_text         56
jo

,job_id,title,company,location,posted_date,posted_text,job_url,seniority_level,employment_type,job_function,industries,description,search_keyword,search_location,scraped_at
0,4446724293,Senior Data Analyst,EXL,"Bengaluru, Karnataka, India",2026-10-02,1 day ago,https://in.linkedin.com/jobs/view/senior-data-...,Mid-Senior level,Full-time,NaN,Business Consulting and Services,Title : Senior Data AnalystLocation : Bangalor...,Data Analyst,India,2026-10-03 22:38:47
1,4433474003,Senior Data Analyst,Finkraft,"Bangalore Urban, Karnataka, India",2026-06-27,3 months ago,https://in.linkedin.com/jobs/view/senior-data-...,Mid-Senior level,Full-time,NaN,Financial Services,About Finkraft.aiFinkraft.ai is a fast-growing...,Data Analyst,India,2026-10-03 22:38:51
2,4471438516,Senior Data Analyst,Guidehouse,"Chennai, Tamil Nadu, India",2026-09-24,1 week ago,https://in.linkedin.com/jobs/view/senior-data-...,Associate,Full-time,NaN,Business Consulting and Services,Job FamilyEBO Management Information System (I...,Data Analyst,India,2026-10-03 22:38:55


In [5]:
# Cell 2: drop the empty column, remove duplicates, clean the text
df = raw.copy()

# 1. drop the column that is empty for every row
df = df.drop(columns=["job_function"])

# 2. remove duplicate jobs (same job_id)
before = len(df)
df = df.drop_duplicates(subset="job_id", keep="first")
print("Rows:", before, "->", len(df))

# 3. clean the text columns: decode HTML (&amp; -> &), trim spaces, collapse repeated spaces
text_cols = ["title", "company", "location", "seniority_level", "employment_type",
             "industries", "description", "search_keyword", "search_location"]

print("'&amp;' found before cleaning:",
      sum(df[c].str.contains("&amp;", na=False).sum() for c in text_cols))

for col in text_cols:
    s = df[col].map(lambda x: html.unescape(x) if isinstance(x, str) else x)
    df[col] = s.str.strip().str.replace(r"\s+", " ", regex=True)

print("'&amp;' found after cleaning: ",
      sum(df[c].str.contains("&amp;", na=False).sum() for c in text_cols))
df[["title", "company", "location"]].head()

Rows: 332 -> 332
'&amp;' found before cleaning: 1
'&amp;' found after cleaning:  0


,title,company,location
0,Senior Data Analyst,EXL,"Bengaluru, Karnataka, India"
1,Senior Data Analyst,Finkraft,"Bangalore Urban, Karnataka, India"
2,Senior Data Analyst,Guidehouse,"Chennai, Tamil Nadu, India"
3,Senior Data Analyst I with Python Experience,ICE,"Pune Division, Maharashtra, India"
4,Assistant Manager,EXL,"Bengaluru, Karnataka, India"


In [6]:
# Cell 3: dates, city and country, remote flag

# ---- 1. dates ----
df["posted_date"] = pd.to_datetime(df["posted_date"], errors="coerce")
df["scraped_at"] = pd.to_datetime(df["scraped_at"], errors="coerce")
df["days_since_posted"] = (df["scraped_at"].dt.normalize() - df["posted_date"]).dt.days

# ---- 2. location -> city + country ----
parts = df["location"].str.split(",")
single = parts.str.len() == 1                                            # no commas, e.g. "India"
is_area = single & df["location"].str.contains(r"\b(?:Area|Region)$", regex=True)   # "Greater Chennai Area"

df["city"] = parts.str[0].str.strip()
df["country"] = parts.str[-1].str.strip()
df.loc[single & ~is_area, "city"] = pd.NA                                # only a country was given
df.loc[is_area, "country"] = pd.NA                                       # it is a city/region, not a country
df["country"] = df["country"].fillna(df["search_location"])             # all our searches were for India

# tidy city names: "Pune City" / "Pune/Pimpri-Chinchwad" -> "Pune", "Bengaluru East" -> "Bengaluru"
df["city"] = (df["city"].str.split("/").str[0]
              .str.replace(r"^Greater\s+", "", regex=True)
              .str.replace(r"\s+(?:Metropolitan Region|Region|Area|Division|Urban|District|Rural|City|East|West|North|South|Central)$",
                           "", regex=True)
              .str.strip())
CITY_FIX = {"Bangalore": "Bengaluru", "Gurgaon": "Gurugram", "Bombay": "Mumbai", "Madras": "Chennai",
            "Calcutta": "Kolkata", "Trivandrum": "Thiruvananthapuram", "Cochin": "Kochi",
            "Poona": "Pune", "New Delhi": "Delhi"}
df["city"] = df["city"].replace(CITY_FIX)

# a state is not a city
STATES = {"Maharashtra", "Karnataka", "Telangana", "Tamil Nadu", "Kerala", "Gujarat", "Delhi NCR",
          "Uttar Pradesh", "West Bengal", "Rajasthan", "Haryana", "Punjab", "Madhya Pradesh",
          "Andhra Pradesh", "Odisha", "Bihar", "Goa"}
df.loc[df["city"].isin(STATES), "city"] = pd.NA

# ---- 3. remote flag ----
df["is_remote"] = (df["title"] + " " + df["location"]).str.contains(
    r"remote|work from home|wfh", case=False, regex=True)

# ---- checks ----
print("Posted date range:", df["posted_date"].min().date(), "->", df["posted_date"].max().date())
print("Missing posted_date:", df["posted_date"].isna().sum())
print("Jobs older than 90 days:", (df["days_since_posted"] > 90).sum())
print("\nTop 20 cities:")
print(df["city"].value_counts().head(20))
print("\nCountries:")
print(df["country"].value_counts())
print("\nRemote jobs:", df["is_remote"].sum(), "of", len(df))

Posted date range: 2025-04-09 -> 2026-10-03
Missing posted_date: 0
Jobs older than 90 days: 26

Top 20 cities:
city
Bengaluru             58
Hyderabad             58
Pune                  49
Chennai               44
Mumbai                33
Gurugram              19
Kochi                 11
Noida                 10
Ahmedabad              8
Kolkata                7
Delhi                  6
Thiruvananthapuram     6
Coimbatore             4
Vadodara               4
Navi Mumbai            3
Indore                 1
Ajmer                  1
Surat                  1
Chandigarh             1
Bhubaneswar            1
Name: count, dtype: int64

Countries:
country
India    332
Name: count, dtype: int64

Remote jobs: 5 of 332


In [7]:
# Cell 4: role category and years of experience

# ---- 1. role category from the job title (the rule that matches EARLIEST in the title wins) ----
ROLE_RULES = [
    ("Machine Learning Engineer", r"machine learning|\bml engineer|\bai engineer|deep learning"),
    ("Data Scientist",            r"data scien"),
    ("Data Engineer",             r"data engineer|\betl\b|big data|data platform|databricks|data ingestion"),
    ("BI Developer",              r"power ?bi|\bbi\b|business intelligence|\bmsbi\b|tableau|\bpbi\b"),
    ("Data Analyst",              r"data analy"),
    ("Business Analyst",          r"business (?:systems? )?analy|systems analyst|\bba\b"),
    ("Software Engineer",         r"software|developer|full ?stack|backend|frontend|programmer"),
]
ROLE_RX = [(name, re.compile(p, re.IGNORECASE)) for name, p in ROLE_RULES]


def role_category(title):
    best, pos = "Other", 10**9
    for name, rx in ROLE_RX:
        m = rx.search(str(title))
        if m and m.start() < pos:
            best, pos = name, m.start()
    return best


df["role_category"] = df["title"].apply(role_category)

# ---- 2. minimum years of experience the JOB asks for ----
EXP_RX = re.compile(r"(?<!\d)(\d{1,2})(?!\d)\s*\+?\s*(?:(?:-|–|to)\s*\d{1,2}\s*\+?\s*)?(?:years?|yrs?)\b",
                    re.IGNORECASE)
# sentences about the COMPANY ("We have 30 years of experience...") must be ignored
COMPANY_RX = re.compile(r"\bwe (?:have|are|'ve|bring|provide|offer)\b"
                        r"|\bour (?:platform|team|company|clients|customers|products?)\b"
                        r"|\bfounded\b|\bsince (?:19|20)\d\d\b|\bproviding\b", re.IGNORECASE)


def min_experience(text):
    if not isinstance(text, str):
        return np.nan
    for sentence in re.split(r"[.;•\n]", text):
        if "experience" in sentence.lower() and not COMPANY_RX.search(sentence):
            m = EXP_RX.search(sentence)
            if m:
                years = int(m.group(1))
                return years if years <= 30 else np.nan
    return np.nan


df["exp_min_years"] = df["description"].apply(min_experience)

# ---- checks ----
print("Role categories:")
print(df["role_category"].value_counts(), "\n")
print("Jobs where experience was found:", df["exp_min_years"].notna().sum(), "of", len(df))
print(df["exp_min_years"].describe().round(1), "\n")
print("Jobs asking for 12+ years:")
print(df.loc[df["exp_min_years"] >= 12, ["title", "exp_min_years"]].to_string())
print("\nTitles classified as 'Other':")
print(df.loc[df["role_category"] == "Other", "title"].to_string())

Role categories:
role_category
Business Analyst     84
Data Scientist       66
Data Engineer        60
Data Analyst         58
BI Developer         53
Other                 7
Software Engineer     4
Name: count, dtype: int64 

Jobs where experience was found: 229 of 332
count    229.0
mean       4.8
std        2.1
min        0.0
25%        3.0
50%        5.0
75%        6.0
max       10.0
Name: exp_min_years, dtype: float64 

Jobs asking for 12+ years:
Empty DataFrame
Columns: [title, exp_min_years]
Index: []

Titles classified as 'Other':
4                 Assistant Manager
24                   Senior Analyst
48                   Senior Analyst
226    Product Definition Analyst 4
243                 Process Manager
310                 Data Consultant
316                 Data Consultant


In [8]:
# Cell 5: extract skills from the descriptions
# Each skill: (regex pattern, ignore_case). You can add your own skills at the end of this dictionary.
SKILL_PATTERNS = {
    "Python": (r"\bpython\b", True),
    "SQL": (r"\bsql\b|\bmysql\b|\bpostgres(?:ql)?\b|\bt-sql\b|\bpl/sql\b", True),
    "R": (r"(?:[,/(:]\s*|\b(?:[Aa]nd|[Oo]r|[Ww]ith|[Uu]sing)\s+)R(?=\s*[,/)]|\s+(?:[Aa]nd|[Oo]r)\b|\s*[.;:]|\s*$)"
          r"|\bR\s+(?:programming|language|studio|packages?|shiny)\b|\bRStudio\b|\bin R\b", False),
    "Java": (r"\bjava\b(?!\s*script)", True),
    "Scala": (r"\bscala\b", True),
    "JavaScript": (r"\bjavascript\b", True),
    "Excel": (r"(?<!\bto )\bexcel\b(?!\s+(?:in|at)\b)", True),
    "Power BI": (r"power\s?bi|\bpbi\b", True),
    "DAX": (r"\bdax\b", True),
    "Power Query": (r"power query", True),
    "Tableau": (r"\btableau\b", True),
    "Looker": (r"\blooker\b", True),
    "Qlik": (r"\bqlik", True),
    "SSIS/SSRS": (r"\bssis\b|\bssrs\b|\bssas\b", True),
    "Pandas": (r"\bpandas\b", True),
    "NumPy": (r"\bnumpy\b", True),
    "Scikit-learn": (r"scikit-?learn|sklearn", True),
    "TensorFlow": (r"tensorflow", True),
    "PyTorch": (r"pytorch", True),
    "Machine Learning": (r"machine learning", True),
    "Deep Learning": (r"deep learning", True),
    "NLP": (r"\bnlp\b|natural language", True),
    "GenAI / LLM": (r"generative ai|\bgenai\b|\bllms?\b", True),
    "Statistics": (r"statistic", True),
    "Data Visualization": (r"data visuali[sz]ation", True),
    "Data Modeling": (r"data model", True),
    "Data Warehousing": (r"data warehous", True),
    "ETL": (r"\betl\b|\belt\b", True),
    "Spark": (r"\bspark\b|pyspark", True),
    "Hadoop": (r"\bhadoop\b", True),
    "Kafka": (r"\bkafka\b", True),
    "Airflow": (r"\bairflow\b", True),
    "dbt": (r"\bdbt\b", True),
    "Databricks": (r"databricks", True),
    "Snowflake": (r"\bsnowflake\b", True),
    "Azure Data Factory": (r"azure data factory|\badf\b", True),
    "Microsoft Fabric": (r"microsoft fabric|ms fabric|onelake", True),
    "AWS": (r"\baws\b|amazon web services", True),
    "Azure": (r"\bazure\b", True),
    "GCP": (r"\bgcp\b|google cloud|\bbigquery\b", True),
    "Docker": (r"\bdocker\b", True),
    "Kubernetes": (r"kubernetes|\bk8s\b", True),
    "Git": (r"\bgit\b|\bgithub\b", True),
    "Agile": (r"\bagile\b|\bscrum\b", True),
    "Jira": (r"\bjira\b", True),
    "Communication": (r"communication", True),
    "Problem Solving": (r"problem[- ]solving", True),
    "Stakeholder Management": (r"stakeholder", True),
}
SKILL_RX = {name: re.compile(p, re.IGNORECASE if ignore else 0)
            for name, (p, ignore) in SKILL_PATTERNS.items()}

GLUE = re.compile(r"(?<=[a-z])(?=[A-Z])")      # finds spots where two words were glued together


def extract_skills(text):
    if not isinstance(text, str):
        return []
    spaced = GLUE.sub(" ", text)                # same text with the glued words separated
    return [name for name, rx in SKILL_RX.items() if rx.search(text) or rx.search(spaced)]


df["skills_list"] = df["description"].apply(extract_skills)
df["skill_count"] = df["skills_list"].str.len()

# one row per (job, skill) - the shape Power BI needs for skill charts
job_skills = (df[["job_id", "skills_list"]]
              .explode("skills_list")
              .dropna(subset=["skills_list"])
              .rename(columns={"skills_list": "skill"})
              .reset_index(drop=True))

df["skills"] = df["skills_list"].apply("; ".join)      # lists don't survive CSV, so join into text
df = df.drop(columns=["skills_list"])

print("Jobs with at least one skill:", (df["skill_count"] > 0).sum(), "of", len(df))
print("Average skills per job:", round(df["skill_count"].mean(), 1))
print("Rows in job_skills table:", len(job_skills))
print("\nTop 20 skills (number of jobs):")
print(job_skills["skill"].value_counts().head(20))

Jobs with at least one skill: 326 of 332
Average skills per job: 8.7
Rows in job_skills table: 2891

Top 20 skills (number of jobs):
skill
Stakeholder Management    228
SQL                       217
Communication             204
Python                    159
Problem Solving           147
Power BI                  138
Data Modeling             130
Azure                     119
Statistics                105
Agile                      98
ETL                        87
Spark                      77
Data Visualization         73
Databricks                 73
Machine Learning           72
AWS                        69
Tableau                    62
Data Warehousing           62
DAX                        60
Snowflake                  59
Name: count, dtype: int64


In [9]:
# Cell 6: label each skill with a category and a type (Technical or Soft)
SKILL_CATEGORY = {
    # Programming languages
    "Python": "Programming", "SQL": "Programming", "R": "Programming", "Java": "Programming",
    "Scala": "Programming", "JavaScript": "Programming",
    # BI and visualization
    "Excel": "BI & Visualization", "Power BI": "BI & Visualization", "DAX": "BI & Visualization",
    "Power Query": "BI & Visualization", "Tableau": "BI & Visualization", "Looker": "BI & Visualization",
    "Qlik": "BI & Visualization", "SSIS/SSRS": "BI & Visualization", "Data Visualization": "BI & Visualization",
    # Data science and machine learning
    "Pandas": "Data Science & ML", "NumPy": "Data Science & ML", "Scikit-learn": "Data Science & ML",
    "TensorFlow": "Data Science & ML", "PyTorch": "Data Science & ML", "Machine Learning": "Data Science & ML",
    "Deep Learning": "Data Science & ML", "NLP": "Data Science & ML", "GenAI / LLM": "Data Science & ML",
    "Statistics": "Data Science & ML",
    # Data engineering
    "Data Modeling": "Data Engineering", "Data Warehousing": "Data Engineering", "ETL": "Data Engineering",
    "Spark": "Data Engineering", "Hadoop": "Data Engineering", "Kafka": "Data Engineering",
    "Airflow": "Data Engineering", "dbt": "Data Engineering", "Databricks": "Data Engineering",
    "Snowflake": "Data Engineering", "Azure Data Factory": "Data Engineering", "Microsoft Fabric": "Data Engineering",
    # Cloud and DevOps
    "AWS": "Cloud & DevOps", "Azure": "Cloud & DevOps", "GCP": "Cloud & DevOps",
    "Docker": "Cloud & DevOps", "Kubernetes": "Cloud & DevOps", "Git": "Cloud & DevOps",
    # Ways of working
    "Agile": "Methods & Collaboration", "Jira": "Methods & Collaboration",
    # Soft skills
    "Communication": "Soft Skill", "Problem Solving": "Soft Skill", "Stakeholder Management": "Soft Skill",
}

job_skills["skill_category"] = job_skills["skill"].map(SKILL_CATEGORY)
job_skills["skill_type"] = np.where(job_skills["skill_category"] == "Soft Skill", "Soft", "Technical")

# safety check: every skill must have a category
unmapped = job_skills.loc[job_skills["skill_category"].isna(), "skill"].unique()
print("Skills without a category:", list(unmapped))

# number of TECHNICAL skills per job, added to the main table
tech_counts = job_skills[job_skills["skill_type"] == "Technical"].groupby("job_id").size()
df["tech_skill_count"] = df["job_id"].map(tech_counts).fillna(0).astype(int)

print("\nSkills per category (distinct jobs):")
print(job_skills.groupby("skill_category")["job_id"].nunique().sort_values(ascending=False))

print("\nTop 15 TECHNICAL skills:")
print(job_skills.loc[job_skills["skill_type"] == "Technical", "skill"].value_counts().head(15))

print("\nAverage technical skills per job:", round(df["tech_skill_count"].mean(), 1))

Skills without a category: []

Skills per category (distinct jobs):
skill_category
Soft Skill                 275
Programming                242
Data Engineering           202
BI & Visualization         183
Cloud & DevOps             152
Data Science & ML          134
Methods & Collaboration    105
Name: job_id, dtype: int64

Top 15 TECHNICAL skills:
skill
SQL                   217
Python                159
Power BI              138
Data Modeling         130
Azure                 119
Statistics            105
Agile                  98
ETL                    87
Spark                  77
Data Visualization     73
Databricks             73
Machine Learning       72
AWS                    69
Tableau                62
Data Warehousing       62
Name: count, dtype: int64

Average technical skills per job: 7.0


In [10]:
# Cell 7: final validation and save

# ---- 1. drop a column we no longer need, put the columns in a sensible order ----
df = df.drop(columns=["posted_text"], errors="ignore")
final_cols = ["job_id", "title", "role_category", "company", "location", "city", "country", "is_remote",
              "posted_date", "days_since_posted", "seniority_level", "employment_type", "industries",
              "exp_min_years", "skill_count", "tech_skill_count", "skills", "job_url", "description",
              "search_keyword", "search_location", "scraped_at"]
df = df[final_cols].sort_values("posted_date", ascending=False).reset_index(drop=True)

# ---- 2. check dimensions ----
print("Main dataset shape:", df.shape)
print("Job skills dataset shape:", job_skills.shape)

# ---- 3. check missing values ----
print("\nMissing values in main dataset:")
print(df.isna().sum())

# ---- 4. check duplicates ----
print("\nDuplicate job IDs:", df["job_id"].duplicated().sum())
print("Duplicate job-skill records:", job_skills.duplicated(subset=["job_id", "skill"]).sum())

# ---- 5. every skill row must belong to a job in the main table ----
orphans = set(job_skills["job_id"]) - set(df["job_id"])
print("Skill rows pointing to unknown jobs:", len(orphans))

# ---- 6. save into data/cleaned_data ----
df.to_csv(CLEAN_DIR / "jobs_cleaned.csv", index=False)
job_skills.to_csv(CLEAN_DIR / "job_skills.csv", index=False)

print("\nSaved files:")
for name in ["jobs_cleaned.csv", "job_skills.csv"]:
    path = CLEAN_DIR / name
    print(f"  {name:<20} exists: {path.exists()}   {path.stat().st_size / 1024:.1f} KB")

print("\nMain dataset preview:")
display(df.head())
print("\nJob skills preview:")
display(job_skills.head())

Main dataset shape: (332, 22)
Job skills dataset shape: (2891, 4)

Missing values in main dataset:
job_id                 0
title                  0
role_category          0
company                0
location               0
city                   2
country                0
is_remote              0
posted_date            0
days_since_posted      0
seniority_level        0
employment_type        0
industries             0
exp_min_years        103
skill_count            0
tech_skill_count       0
skills                 0
job_url                0
description            0
search_keyword         0
search_location        0
scraped_at             0
dtype: int64

Duplicate job IDs: 0
Duplicate job-skill records: 0
Skill rows pointing to unknown jobs: 0

Saved files:
  jobs_cleaned.csv     exists: True   1304.0 KB
  job_skills.csv       exists: True   133.2 KB

Main dataset preview:


,job_id,title,role_category,company,location,city,country,is_remote,posted_date,days_since_posted,...,industries,exp_min_years,skill_count,tech_skill_count,skills,job_url,description,search_keyword,search_location,scraped_at
0,4475050785,DATA ANALYST L4,Data Analyst,Wipro,"Hyderabad, Telangana, India",Hyderabad,India,False,2026-10-03,0,...,IT Services and IT Consulting,5.0,2,2,Data Warehousing; GCP,https://in.linkedin.com/jobs/view/data-analyst...,Job description:Job Description Role PurposeTh...,Data Analyst,India,2026-10-03 22:39:53
1,4466303542,Principal Data Scientist-GenAI l LLMs l Comput...,Data Scientist,HERE Technologies,Mumbai Metropolitan Region,Mumbai,India,False,2026-10-03,0,...,IT Services and IT Consulting,10.0,8,8,Python; SQL; R; Machine Learning; Deep Learnin...,https://in.linkedin.com/jobs/view/principal-da...,What's the role? The Principal Data Scientist ...,Data Scientist,India,2026-10-03 23:00:39
2,4446723624,Associate - Business Analyst-Business Process ...,Business Analyst,EXL,"Noida, Uttar Pradesh, India",Noida,India,False,2026-10-03,0,...,Business Consulting and Services,NaN,0,0,,https://in.linkedin.com/jobs/view/associate-bu...,Associate Business Analysts support business a...,Business Analyst,India,2026-10-03 23:12:19
3,4446446065,Business Analyst,Business Analyst,Flex,"Chennai, Tamil Nadu, India",Chennai,India,False,2026-10-03,0,...,"Appliances, Electrical, and Electronics Manufa...",NaN,4,1,SQL; Communication; Problem Solving; Stakehold...,https://in.linkedin.com/jobs/view/business-ana...,Flex is the diversified manufacturing partner ...,Business Analyst,India,2026-10-03 23:12:01
4,4466089990,Senior Business Systems Analyst,Business Analyst,Franklin Templeton,Greater Hyderabad Area,Hyderabad,India,False,2026-10-03,0,...,Financial Services,6.0,4,2,Agile; Jira; Communication; Stakeholder Manage...,https://in.linkedin.com/jobs/view/senior-busin...,"At Franklin Templeton, we believe success is b...",Business Analyst,India,2026-10-03 23:15:35



Job skills preview:


,job_id,skill,skill_category,skill_type
0,4446724293,SQL,Programming,Technical
1,4446724293,Power BI,BI & Visualization,Technical
2,4446724293,Tableau,BI & Visualization,Technical
3,4446724293,Stakeholder Management,Soft Skill,Soft
4,4433474003,Python,Programming,Technical


In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SQL_DIR = ROOT / "data" / "sql_ready"
SQL_DIR.mkdir(exist_ok=True)


def plain(s):
    """Replace common special characters, then drop anything that is not plain ASCII."""
    if not isinstance(s, str):
        return s
    for bad, good in {"–": "-", "—": "-", "’": "'", "‘": "'", "“": '"', "”": '"'}.items():
        s = s.replace(bad, good)
    return s.encode("ascii", "ignore").decode()


for name in ["jobs_cleaned", "job_skills"]:
    df = pd.read_csv(ROOT / "data" / "cleaned_data" / f"{name}.csv", dtype={"job_id": str})
    df = df.drop(columns=["description"], errors="ignore")
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]) and not pd.api.types.is_bool_dtype(df[col]):
            df[col] = df[col].map(plain)
    df.to_csv(SQL_DIR / f"{name}.csv", index=False)
    print(name, df.shape)

jobs_cleaned (332, 21)
job_skills (2891, 4)
